# Prior knowledge versus new evidence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamza11235/Impact-of-prior-knowledge-on-discovery/blob/main/notebooks/prior_evidence_demo.ipynb)

This notebook is the reviewer-facing demonstration for the first experimental block.

**Question:** if a fictional scientific law is stated as established background knowledge,
will Qwen3-8B still infer a conflicting law from clean numerical observations?

The notebook has two modes:

- **Cached mode (default):** loads the checked-in prompts and all 130 saved generation
  records. It requires no model download and reproduces the reported tables.
- **Live mode (optional):** runs one selected response through Qwen3-8B using MLX on
  Apple silicon.

The notebook imports the experiment implementation from `src/prior_evidence`; it does
not contain a separate copy of the scientific logic.

After cloning the repository, the complete cached demonstration runs with:

```bash
python -m pip install -e ".[notebook]"
jupyter lab notebooks/prior_evidence_demo.ipynb
```

No credentials, model weights, or external services are required in cached mode.

In [1]:
# Fresh Colab sessions open only the notebook, so fetch its accompanying
# source code and experiment artifacts automatically. This cell is a no-op after cloning
# the repository locally.
import os
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = (
    "https://github.com/hamza11235/"
    "Impact-of-prior-knowledge-on-discovery.git"
)
if "google.colab" in sys.modules:
    colab_root = Path("/content/Impact-of-prior-knowledge-on-discovery")
    if not (colab_root / "pyproject.toml").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(colab_root)],
            check=True,
        )
    os.chdir(colab_root)
    print(f"Colab repository ready: {colab_root}")

In [2]:
from pathlib import Path
import hashlib
import json
import math
import sys

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    candidates = (start.resolve(), *start.resolve().parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "src" / "prior_evidence"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "src"
EXPERIMENT_DIR = ROOT / "experiments" / "01_in_context_vs_base"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Reviewer controls. Cached mode works without a model download.
MODE = "cached"  # "cached" or "live_mlx"
DATASET_SEED = 2000
TARGET_LAW = "L_B"  # "L_A" or "L_B"
PRIOR_ARM = "bare"  # "none", "guided", or "bare"

# Optionally paste a response from any other model and score it with this benchmark.
# When set to a non-empty string, this takes precedence over MODE.
PASTED_RESPONSE = None

if MODE not in {"cached", "live_mlx"}:
    raise ValueError("MODE must be 'cached' or 'live_mlx'")
if TARGET_LAW not in {"L_A", "L_B"}:
    raise ValueError("TARGET_LAW must be 'L_A' or 'L_B'")
if PRIOR_ARM not in {"none", "guided", "bare"}:
    raise ValueError("PRIOR_ARM must be 'none', 'guided', or 'bare'")

print(f"Repository: {ROOT.name}")
print(
    f"Mode={MODE}, seed={DATASET_SEED}, target={TARGET_LAW}, "
    f"prior_arm={PRIOR_ARM}"
)

Repository: prior-evidence-diagnostic
Mode=cached, seed=2000, target=L_B, prior_arm=bare


## 1. Controlled fictional laws

The model is told that the quantities belong to a fictional neryx domain. The prior law is

$$
L_A:\qquad \tau = 3m^1r^2q^{-1}s^0,
$$

while the default conflicting observations are generated by

$$
L_B:\qquad \tau = 3m^1r^1q^{-2}s^0.
$$

Fictional names reduce direct lexical recall of a named real-world law. They do not remove
the model's general mathematical knowledge about ratios and power laws. Change
`TARGET_LAW` in the configuration cell to generate observations from either law.

In [3]:
from prior_evidence.domain import (
    LAW_A,
    LAW_B,
    VARIABLES,
    generate_evidence,
    generate_isolating_evidence,
)
from prior_evidence.metrics import exponent_l1_error, heldout_log_mse, law_matches
from prior_evidence.parsing import HypothesisParseError, parse_hypothesis
from prior_evidence.prompting import build_messages

LAW_BY_NAME = {"L_A": LAW_A, "L_B": LAW_B}
target_law = LAW_BY_NAME[TARGET_LAW]
dataset = generate_isolating_evidence(
    law=target_law,
    sigma=0.0,
    seed=DATASET_SEED,
)


def changed_variable(row) -> str:
    changed = [
        variable
        for variable, value in zip(VARIABLES, row.inputs, strict=True)
        if value != 1.0
    ]
    return changed[0] if changed else "baseline"


order = {"baseline": 0, "m": 1, "r": 2, "q": 3, "s": 4}
ordered_rows = sorted(
    dataset.observations,
    key=lambda row: (order[changed_variable(row)], row.inputs),
)

table = [
    "| sweep | m | r | q | s | tau |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in ordered_rows:
    table.append(
        f"| {changed_variable(row)} | {row.m:.6g} | {row.r:.6g} | "
        f"{row.q:.6g} | {row.s:.6g} | {row.tau:.6g} |"
    )

display(Markdown("\n".join(table)))

| sweep | m | r | q | s | tau |
|---|---:|---:|---:|---:|---:|
| baseline | 1 | 1 | 1 | 1 | 3 |
| m | 0.372416 | 1 | 1 | 1 | 1.11725 |
| m | 4.64686 | 1 | 1 | 1 | 13.9406 |
| r | 1 | 0.386093 | 1 | 1 | 1.15828 |
| r | 1 | 3.62604 | 1 | 1 | 10.8781 |
| q | 1 | 1 | 0.598688 | 1 | 8.3699 |
| q | 1 | 1 | 4.56347 | 1 | 0.144056 |
| s | 1 | 1 | 1 | 0.284438 | 3 |
| s | 1 | 1 | 1 | 3.76792 | 3 |

Each non-baseline row changes exactly one variable. This makes an exponent directly
recoverable from a ratio:

$$
\frac{\tau_2}{\tau_1}
=
\left(\frac{x_2}{x_1}\right)^{\alpha_x},
\qquad
\alpha_x
=
\frac{\log(\tau_2/\tau_1)}{\log(x_2/x_1)}.
$$

The formula is shown here to explain the design. It is **not included in the model
prompt**.

In [4]:
baseline = next(
    row for row in dataset.observations if changed_variable(row) == "baseline"
)
q_row = next(
    row
    for row in dataset.observations
    if changed_variable(row) == "q" and row.q < 1
)
inferred_q_exponent = math.log(q_row.tau / baseline.tau) / math.log(
    q_row.q / baseline.q
)

display(
    Markdown(
        f"For the low-$q$ intervention, $q={q_row.q:.4f}$ and "
        f"$\\tau={q_row.tau:.4f}$. Relative to the baseline, this gives "
        f"$\\alpha_q={inferred_q_exponent:.6f}$. The configured target has "
        f"exact exponent ${target_law.exponents[2]:g}$."
    )
)

For the low-$q$ intervention, $q=0.5987$ and $\tau=8.3699$. Relative to the baseline, this gives $\alpha_q=-2.000000$. The configured target has exact exponent $-2$.

## 2. Three prompt arms

All arms receive the same configured observations and the same hypothesis class. The
checked-in comparison uses seeds 2000 through 2009 with $L_B$ observations.

1. **No prior:** the model sees only the observations.
2. **Guided in-context prior:** $L_A$ is stated, followed by explicit guidance that the
   observations may confirm or contradict it.
3. **Bare in-context prior:** $L_A$ is stated as established background knowledge, with
   no warning that it may be wrong and no instruction to override it.

The bare arm addresses the concern that the guided wording itself encouraged rejection of
the prior. The configuration cell selects which arm will be loaded or run below.

In [5]:
ARM_CONFIG_TO_INTERNAL = {
    "none": "none",
    "guided": "in_context",
    "bare": "in_context_bare",
}
ARM_LABELS = {
    "none": "No prior",
    "in_context": "Guided in-context prior",
    "in_context_bare": "Bare in-context prior",
}
selected_internal_arm = ARM_CONFIG_TO_INTERNAL[PRIOR_ARM]
messages_by_arm = {
    arm: build_messages(dataset, prior_arm=arm)
    for arm in ARM_LABELS
}

for arm, label in ARM_LABELS.items():
    messages = messages_by_arm[arm]
    rendered = "\n\n".join(
        f"[{message['role'].upper()}]\n{message['content']}" for message in messages
    )
    display(
        Markdown(
            f"<details><summary><strong>{label}: exact prompt</strong></summary>\n\n"
            f"```text\n{rendered}\n```\n\n</details>"
        )
    )

print(f"Selected arm: {ARM_LABELS[selected_internal_arm]}")

<details><summary><strong>No prior: exact prompt</strong></summary>

```text
[SYSTEM]
You are analyzing measurements from a fictional quantitative domain.
Use only the supplied observations and the stated hypothesis class.
Do not substitute a familiar real-world law.
Return exactly one valid JSON object and no surrounding prose or Markdown.


[USER]
In this fictional domain:
- m is velic mass
- r is orbin radius
- q is quorin density
- s is selan index
- tau is neryx period

Assume that, if a stable law is present, it belongs to this family:

tau = c * m^alpha_m * r^alpha_r * q^alpha_q * s^alpha_s

Here are 9 observations:

Baseline:
m,r,q,s,tau

1,1,1,1,3

Sweep for m (only m changes):
m,r,q,s,tau

0.37241574,1,1,1,1.1172472
4.6468579,1,1,1,13.940574

Sweep for r (only r changes):
m,r,q,s,tau

1,0.38609253,1,1,1.1582776
1,3.626038,1,1,10.878114

Sweep for q (only q changes):
m,r,q,s,tau

1,1,0.59868809,1,8.3698952
1,1,4.5634673,1,0.144056

Sweep for s (only s changes):
m,r,q,s,tau

1,1,1,0.28443825,3
1,1,1,3.7679238,3

These are controlled one-variable-at-a-time measurements.
Before returning the final JSON, inspect the baseline and the m, r, q, and s sweeps
separately. Do not skip a sweep.

Infer c and the four exponents from the observations.

Return exactly this JSON shape:
{
  "no_law_recoverable": <true or false>,
  "constant": <positive number or null>,
  "exponents": <null or {
    "m": <number>,
    "r": <number>,
    "q": <number>,
    "s": <number>
  }>
}

```

</details>

<details><summary><strong>Guided in-context prior: exact prompt</strong></summary>

```text
[SYSTEM]
You are analyzing measurements from a fictional quantitative domain.
Use only the supplied background knowledge, observations, and stated hypothesis class.
Do not substitute a familiar real-world law.
Return exactly one valid JSON object and no surrounding prose or Markdown.


[USER]
In this fictional domain:
- m is velic mass
- r is orbin radius
- q is quorin density
- s is selan index
- tau is neryx period

Established background knowledge for this fictional domain:

tau = 3 * m^1 * r^2 * q^-1 * s^0

Use this as prior knowledge when analyzing the new observations. The observations may
confirm or contradict the background law. Report the law best supported by the supplied
observations.

Assume that, if a stable law is present, it belongs to this family:

tau = c * m^alpha_m * r^alpha_r * q^alpha_q * s^alpha_s

Here are 9 observations:

Baseline:
m,r,q,s,tau

1,1,1,1,3

Sweep for m (only m changes):
m,r,q,s,tau

0.37241574,1,1,1,1.1172472
4.6468579,1,1,1,13.940574

Sweep for r (only r changes):
m,r,q,s,tau

1,0.38609253,1,1,1.1582776
1,3.626038,1,1,10.878114

Sweep for q (only q changes):
m,r,q,s,tau

1,1,0.59868809,1,8.3698952
1,1,4.5634673,1,0.144056

Sweep for s (only s changes):
m,r,q,s,tau

1,1,1,0.28443825,3
1,1,1,3.7679238,3

These are controlled one-variable-at-a-time measurements.
Before returning the final JSON, inspect the baseline and the m, r, q, and s sweeps
separately. Do not skip a sweep.

Infer c and the four exponents from the observations.

Return exactly this JSON shape:
{
  "no_law_recoverable": <true or false>,
  "constant": <positive number or null>,
  "exponents": <null or {
    "m": <number>,
    "r": <number>,
    "q": <number>,
    "s": <number>
  }>
}

```

</details>

<details><summary><strong>Bare in-context prior: exact prompt</strong></summary>

```text
[SYSTEM]
You are analyzing measurements from a fictional quantitative domain.
Use only the supplied background knowledge, observations, and stated hypothesis class.
Do not substitute a familiar real-world law.
Return exactly one valid JSON object and no surrounding prose or Markdown.


[USER]
In this fictional domain:
- m is velic mass
- r is orbin radius
- q is quorin density
- s is selan index
- tau is neryx period

Established background knowledge for this fictional domain:

tau = 3 * m^1 * r^2 * q^-1 * s^0

Assume that, if a stable law is present, it belongs to this family:

tau = c * m^alpha_m * r^alpha_r * q^alpha_q * s^alpha_s

Here are 9 observations:

Baseline:
m,r,q,s,tau

1,1,1,1,3

Sweep for m (only m changes):
m,r,q,s,tau

0.37241574,1,1,1,1.1172472
4.6468579,1,1,1,13.940574

Sweep for r (only r changes):
m,r,q,s,tau

1,0.38609253,1,1,1.1582776
1,3.626038,1,1,10.878114

Sweep for q (only q changes):
m,r,q,s,tau

1,1,0.59868809,1,8.3698952
1,1,4.5634673,1,0.144056

Sweep for s (only s changes):
m,r,q,s,tau

1,1,1,0.28443825,3
1,1,1,3.7679238,3

These are controlled one-variable-at-a-time measurements.
Before returning the final JSON, inspect the baseline and the m, r, q, and s sweeps
separately. Do not skip a sweep.

Infer c and the four exponents from the observations.

Return exactly this JSON shape:
{
  "no_law_recoverable": <true or false>,
  "constant": <positive number or null>,
  "exponents": <null or {
    "m": <number>,
    "r": <number>,
    "q": <number>,
    "s": <number>
  }>
}

```

</details>

Selected arm: Bare in-context prior


In [6]:
guided_user = messages_by_arm["in_context"][1]["content"]
bare_user = messages_by_arm["in_context_bare"][1]["content"]

removed_guidance = (
    "Use this as prior knowledge when analyzing the new observations. The observations may\n"
    "confirm or contradict the background law. Report the law best supported by the supplied\n"
    "observations."
)

print("Guidance present in guided prompt:", removed_guidance in guided_user)
print("Guidance present in bare prompt:  ", removed_guidance in bare_user)

Guidance present in guided prompt: True
Guidance present in bare prompt:   False


## 3. Load and validate the checked-in experiment

The repository includes all raw generation records, rather than only a hand-written
summary:

- 100 feasibility generations;
- 10 no-prior $L_B$ generations;
- 10 guided-prior generations;
- 10 bare-prior generations.

First, verify that the checked-in artifacts match their SHA-256 manifest.

In [7]:
def verify_checksums(experiment_dir: Path) -> list[str]:
    verified = []
    manifest = experiment_dir / "SHA256SUMS"
    for line in manifest.read_text().splitlines():
        expected, relative_path = line.split(maxsplit=1)
        payload = (experiment_dir / relative_path).read_bytes()
        observed = hashlib.sha256(payload).hexdigest()
        if observed != expected:
            raise ValueError(f"Checksum mismatch: {relative_path}")
        verified.append(relative_path)
    return verified


verified = verify_checksums(EXPERIMENT_DIR)
print(f"Verified {len(verified)} checked-in experiment files.")

Verified 17 checked-in experiment files.


In [8]:
def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text().splitlines()
        if line.strip()
    ]


artifact_dir = EXPERIMENT_DIR / "artifacts"
records = {
    "feasibility": load_jsonl(artifact_dir / "feasibility_runs.jsonl"),
    "none": load_jsonl(artifact_dir / "no_prior_law_b_runs.jsonl"),
    "in_context": load_jsonl(artifact_dir / "in_context_guided_runs.jsonl"),
    "in_context_bare": load_jsonl(artifact_dir / "in_context_bare_runs.jsonl"),
}
experiment_summary = json.loads(
    (EXPERIMENT_DIR / "results" / "experiment_summary.json").read_text()
)

print({name: len(rows) for name, rows in records.items()})

{'feasibility': 100, 'none': 10, 'in_context': 10, 'in_context_bare': 10}


## 4. Try one configured case

The notebook now selects the response corresponding to the configuration at the top:

- in `cached` mode, it finds the matching checked-in Qwen trace;
- in `live_mlx` mode, it generates one new trace;
- when `PASTED_RESPONSE` is supplied, it scores that response instead.

Cached $L_B$ traces are available for seeds 2000–2009 in all three prompt arms.

In [9]:
selected_record = None
raw_response = None
response_source = None
response_truncated = None

if PASTED_RESPONSE:
    raw_response = PASTED_RESPONSE
    response_source = "pasted response"
elif MODE == "cached":
    if TARGET_LAW == "L_B":
        candidates = [
            row
            for row in records[selected_internal_arm]
            if row["dataset_seed"] == DATASET_SEED
        ]
        if candidates:
            selected_record = candidates[0]
            raw_response = selected_record["raw_response"]
            response_truncated = selected_record["truncated"]
            response_source = (
                f"cached {ARM_LABELS[selected_internal_arm]} trace, "
                f"seed {DATASET_SEED}"
            )
    if raw_response is None:
        display(
            Markdown(
                "**No cached response exists for this configuration.** "
                "Choose target `L_B` with a seed from 2000 through 2009, "
                "switch to `live_mlx`, or paste a response."
            )
        )
elif MODE == "live_mlx":
    from prior_evidence.backends import MLXBackend

    backend = MLXBackend(
        "mlx-community/Qwen3-8B-4bit",
        enable_thinking=True,
        top_p=0.95,
        top_k=20,
    )
    generation = backend.generate(
        messages_by_arm[selected_internal_arm],
        max_tokens=6144,
        temperature=0.6,
        seed=10_000_000 + DATASET_SEED,
    )
    raw_response = generation.text
    response_truncated = generation.finish_reason == "length" or (
        "<think>" in generation.text and "</think>" not in generation.text
    )
    response_source = "fresh MLX generation"

print(f"Response source: {response_source or 'none'}")

Response source: cached Bare in-context prior trace, seed 2000


In [10]:
def visible_answer(raw: str) -> str:
    return raw.rsplit("</think>", 1)[-1].strip()


if raw_response is not None:
    try:
        parsed = parse_hypothesis(raw_response)
        parse_error = None
    except HypothesisParseError as exc:
        parsed = None
        parse_error = str(exc)

    if parsed is not None:
        heldout = generate_evidence(
            law=target_law,
            n=200,
            sigma=0.0,
            seed=DATASET_SEED + 1_000_000,
        )
        score = {
            "matches_target": law_matches(parsed, target_law),
            "matches_L_A_prior": law_matches(parsed, LAW_A),
            "exponent_L1_error": exponent_l1_error(parsed, target_law),
            "heldout_log_MSE": heldout_log_mse(parsed, heldout),
            "truncated": response_truncated,
        }
    else:
        score = {
            "matches_target": False,
            "matches_L_A_prior": False,
            "exponent_L1_error": None,
            "heldout_log_MSE": None,
            "truncated": response_truncated,
        }

    display(
        Markdown(
            f"### Scored response: {response_source}\n\n"
            f"```json\n{visible_answer(raw_response)}\n```"
        )
    )
    print("Parsed hypothesis:", parsed.to_dict() if parsed is not None else None)
    print("Parse error:", parse_error)
    print(json.dumps(score, indent=2))
else:
    parsed = None
    score = None

### Scored response: cached Bare in-context prior trace, seed 2000

```json
{
  "no_law_recoverable": false,
  "constant": 3,
  "exponents": {
    "m": 1,
    "r": 1,
    "q": -2,
    "s": 0
  }
}
```

Parsed hypothesis: {'no_law_recoverable': False, 'constant': 3.0, 'exponents': {'m': 1.0, 'r': 1.0, 'q': -2.0, 's': 0.0}}
Parse error: None
{
  "matches_target": true,
  "matches_L_A_prior": false,
  "exponent_L1_error": 0.0,
  "heldout_log_MSE": 0.0,
  "truncated": false
}


## 5. Feasibility gate

The prior comparison is meaningful only if the base model can infer a controlled power
law from evidence. The feasibility gate used 20 independently varied $L_A$ datasets and
five samples per dataset.

In [11]:
feasibility_runs = records["feasibility"]
feasibility_dataset_count = len(
    {row["dataset_seed"] for row in feasibility_runs}
)
feasibility = {
    "datasets": feasibility_dataset_count,
    "samples_per_dataset": len(feasibility_runs) // feasibility_dataset_count,
    "total_generations": len(feasibility_runs),
    "valid_exact_outputs": sum(
        not row["truncated"]
        and row["parse_error"] is None
        and row["law_match"]
        for row in feasibility_runs
    ),
    "completed_output_contract_failures": sum(
        not row["truncated"] and row["parse_error"] is not None
        for row in feasibility_runs
    ),
    "truncations_after_correct_derivation": sum(
        row["truncated"] for row in feasibility_runs
    ),
    "scientifically_wrong_parsed_outputs": sum(
        not row["truncated"]
        and row["parse_error"] is None
        and not row["law_match"]
        for row in feasibility_runs
    ),
}

# The summary is a cross-check, not the source of the displayed counts.
assert feasibility == experiment_summary["feasibility"]

feasibility_table = [
    "| quantity | count |",
    "|---|---:|",
    f"| Total generations | {feasibility['total_generations']} |",
    f"| Valid exact outputs | {feasibility['valid_exact_outputs']} |",
    f"| Completed output-contract failures after correct derivation | "
    f"{feasibility['completed_output_contract_failures']} |",
    f"| Truncations after correct derivation | "
    f"{feasibility['truncations_after_correct_derivation']} |",
    f"| Scientifically wrong parsed outputs | "
    f"{feasibility['scientifically_wrong_parsed_outputs']} |",
]
display(Markdown("\n".join(feasibility_table)))

| quantity | count |
|---|---:|
| Total generations | 100 |
| Valid exact outputs | 96 |
| Completed output-contract failures after correct derivation | 2 |
| Truncations after correct derivation | 2 |
| Scientifically wrong parsed outputs | 0 |

The four non-valid generations were not alternative scientific hypotheses: two violated
the requested JSON contract after deriving the correct law, and two reached the token
limit after deriving the correct exponents. Parsed hypotheses had zero exponent error and
zero held-out prediction error.

## 6. Matched in-context comparison

The following mechanical counts are recomputed from the raw JSONL records for ten matched
$L_B$ datasets with seeds 2000–2009. The curated summary is used only as an assertion
that the recomputation agrees with the reported result.

In [12]:
def aggregate_arm(rows: list[dict]) -> dict[str, int]:
    parsed_rows = []
    for row in rows:
        try:
            hypothesis = parse_hypothesis(row["raw_response"])
        except HypothesisParseError:
            hypothesis = None
        parsed_rows.append((row, hypothesis))

    return {
        "total": len(rows),
        "valid_exact_L_B": sum(
            not row["truncated"]
            and hypothesis is not None
            and law_matches(hypothesis, LAW_B)
            for row, hypothesis in parsed_rows
        ),
        "parse_failures": sum(
            not row["truncated"] and hypothesis is None
            for row, hypothesis in parsed_rows
        ),
        "truncations": sum(row["truncated"] for row in rows),
        "L_A_outputs": sum(
            not row["truncated"]
            and hypothesis is not None
            and law_matches(hypothesis, LAW_A)
            for row, hypothesis in parsed_rows
        ),
    }


computed_comparison = {
    arm: aggregate_arm(records[arm])
    for arm in ("none", "in_context", "in_context_bare")
}
reported_comparison = experiment_summary["matched_conflict_comparison"]

assert computed_comparison["none"]["valid_exact_L_B"] == (
    reported_comparison["no_prior"]["valid_exact_l_b_outputs"]
)
assert computed_comparison["in_context"]["valid_exact_L_B"] == (
    reported_comparison["guided_in_context_l_a"]["valid_exact_l_b_outputs"]
)
assert computed_comparison["in_context_bare"]["valid_exact_L_B"] == (
    reported_comparison["bare_in_context_l_a"][
        "completed_valid_exact_l_b_outputs"
    ]
)
assert all(row["L_A_outputs"] == 0 for row in computed_comparison.values())

result_table = [
    "| arm | exact valid L_B | parse failures | truncations | L_A outputs |",
    "|---|---:|---:|---:|---:|",
]
for arm in ("none", "in_context", "in_context_bare"):
    row = computed_comparison[arm]
    result_table.append(
        f"| {ARM_LABELS[arm]} | {row['valid_exact_L_B']}/{row['total']} | "
        f"{row['parse_failures']} | {row['truncations']} | {row['L_A_outputs']} |"
    )

display(Markdown("\n".join(result_table)))

| arm | exact valid L_B | parse failures | truncations | L_A outputs |
|---|---:|---:|---:|---:|
| No prior | 9/10 | 1 | 0 | 0 |
| Guided in-context prior | 10/10 | 0 | 0 | 0 |
| Bare in-context prior | 9/10 | 0 | 1 | 0 |

In [13]:
def visible_answer(raw_response: str) -> str:
    return raw_response.rsplit("</think>", 1)[-1].strip()


for arm in ("none", "in_context", "in_context_bare"):
    example = records[arm][0]
    label = ARM_LABELS[arm]
    final_answer = visible_answer(example["raw_response"])
    display(
        Markdown(
            f"### {label}: representative final answer\n\n"
            f"```json\n{final_answer}\n```\n\n"
            f"- Target-law match: **{example['law_match']}**\n"
            f"- Prior-law match: **{example.get('prior_law_match', False)}**\n"
            f"- Truncated: **{example['truncated']}**"
        )
    )

### No prior: representative final answer

```json
{
  "no_law_recoverable": false,
  "constant": 3,
  "exponents": {
    "m": 1,
    "r": 1,
    "q": -2,
    "s": 0
  }
}
```

- Target-law match: **True**
- Prior-law match: **False**
- Truncated: **False**

### Guided in-context prior: representative final answer

```json
{
  "no_law_recoverable": false,
  "constant": 3,
  "exponents": {
    "m": 1,
    "r": 1,
    "q": -2,
    "s": 0
  }
}
```

- Target-law match: **True**
- Prior-law match: **False**
- Truncated: **False**

### Bare in-context prior: representative final answer

```json
{
  "no_law_recoverable": false,
  "constant": 3,
  "exponents": {
    "m": 1,
    "r": 1,
    "q": -2,
    "s": 0
  }
}
```

- Target-law match: **True**
- Prior-law match: **False**
- Truncated: **False**

## 7. Running a fresh response

The configured-case cells near the top support three response sources:

- `MODE = "cached"` selects a checked-in response for $L_B$, seeds 2000--2009;
- `MODE = "live_mlx"` generates one fresh Qwen3-8B response on Apple silicon;
- `PASTED_RESPONSE = "..."` scores a response copied from another model or service.

For live inference, install `python -m pip install -e ".[mac,notebook]"`, change the
top configuration cell, and rerun the notebook. Only the configured case is generated,
so a reviewer can try the system without repeating the full 130-generation study.

## 8. Interpretation and boundary of the result

Under clean, decisive controlled evidence:

- the base model recovered $L_B$;
- the guided in-context arm recovered $L_B$;
- removing explicit override guidance did not change the scientific conclusion;
- no in-context trace returned the asserted $L_A$ prior.

The supported conclusion is therefore narrow:

> An incompatible law stated in context produced no detectable bias toward that law under
> clean, directly diagnostic evidence.

This is not yet a test of persistent learning. Information supplied in context is
temporary and visible. The next experimental block introduces $L_A$ through LoRA,
checks whether the law is genuinely retrievable, and tests whether the model's original
law-induction capability survives fine-tuning.